# Comparaison of samples

Apply to output files from summary_celltypes_IHOPE.py in CSV format.

In [ ]:
import sys
from pathlib import Path
#TODO try to remove this somehow
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import importlib
import scripts.comparison as comparison

importlib.reload(comparison)

from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    print_numeric_summary,
    pivot_for_tissue_heatmap,
    plot_celltype_heatmap
)


Define file names and sample names. Use the output from cell 1 to complete cell 2.

In [ ]:
from pathlib import Path

summaries_dir = Path("../results/reports/NEW")
files = sorted(summaries_dir.glob("celltype_summary_*.csv"))

print("name_map = {")
for f in files:
    print(f'    "{f.name}": "",')
print("}")

In [ ]:
name_map = {
    "celltype_summary_IHOPE14_MedLN_BottomLeft_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE14_MedLN_BottomLeft",
    "celltype_summary_IHOPE14_MedLN_BottomRight_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE14_MedLN_BottomRight",
    "celltype_summary_IHOPE14_MedLN_TopRight_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE14_MedLN_TopRight",
    "celltype_summary_IHOPE14_mesLN_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE14_mesLN",
    "celltype_summary_IHOPE20_LN_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE20_LN",
    "celltype_summary_IHOPE20_Spleen_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE20_Spleen",
    "celltype_summary_IHOPE26_LN_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE26_LN",
    "celltype_summary_IHOPE26_Spleen_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE26_Spleen",
    "celltype_summary_IHOPE27_LN_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE27_LN",
    "celltype_summary_IHOPE27_Spleen_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE27_Spleen",
    "celltype_summary_IHOPE39_LN_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE39_LN",
    "celltype_summary_IHOPE39_MesLN_1_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE39_MesLN_1",
    "celltype_summary_IHOPE39_Spleen_filtered_arcsinh_cf5.0_IHOPE_NEW_summary.csv": "IHOPE39_Spleen",
}

In [ ]:
file_map = {
    sample: summaries_dir / fname
    for fname, sample in name_map.items()
}

In [ ]:
# Sanity check
for sample, path in file_map.items():
    print(sample, ": ", path.name)

Load data frame

In [ ]:
df = load_celltype_summaries(file_map=file_map)

matrix = pivot_for_heatmap(df)

Plot cell type heatmap

In [ ]:
# Fix the coloring for low-abundant cell types:
import numpy as np
plot_matrix = np.log10(matrix + 0.1)

plot_celltype_heatmap(
    plot_matrix,
    title="Cell type composition (log-scaled) across samples (all levels)",
)

plot_celltype_heatmap(matrix, title="Linear scale")


Tissue type summary

In [ ]:
tissue_map = {
    "IHOPE14_MedLN_BottomLeft": "MedLN",
    "IHOPE14_MedLN_BottomRight": "MedLN",
    "IHOPE14_MedLN_TopRight": "MedLN",
    "IHOPE14_mesLN": "MesLN",
    "IHOPE20_LN": "MedLN",
    "IHOPE20_Spleen": "Spleen",
    "IHOPE26_LN": "MedLN",
    "IHOPE26_Spleen": "Spleen",
    "IHOPE27_LN": "MedLN",
    "IHOPE27_Spleen": "Spleen",
    "IHOPE39_LN": "MedLN",
    "IHOPE39_MesLN_1": "MesLN",
    "IHOPE39_Spleen": "Spleen",
}

In [ ]:
df = load_celltype_summaries(file_map=file_map)

df["tissue"] = df["sample"].map(tissue_map)

Using mean  (avoid dominance of large samples):

In [ ]:
df_tissue = (
    df.groupby(["tissue", "level", "cell_type"], as_index=False)
      .agg(pct_total=("pct_total", "mean"))
)

In [ ]:
matrix_tissue = pivot_for_tissue_heatmap(df_tissue)

plot_matrix = np.log10(matrix_tissue + 0.1)

In [ ]:
plot_celltype_heatmap(
    plot_matrix,
    title="Cell type composition across tissues (log-scaled)",
)